In [2]:
# 定义10条评估 queries
eval_queries = [
    "How should an organization respond to a phishing email?",
    "What technical controls can prevent phishing attacks?",
    "How can employees identify spear-phishing attempts?",
    "What is DMARC and how does it help prevent email spoofing?",
    "How should incident response teams handle a phishing-related data breach?",
    "What are best practices for email authentication in enterprises?",
    "How can organizations train employees to recognize phishing?",
    "What role does SPF play in email security?",
    "How should a mid-sized enterprise handle a ransomware attack initiated by phishing?",
    "What are the indicators of compromise in a phishing attack?"
]

print(f"Total evaluation queries: {len(eval_queries)}")

Total evaluation queries: 10


In [3]:
# 跑完整 RAG pipeline，收集所有结果
# 复制上一个notebook的函数（这样evaluation notebook独立可运行）
import os, requests
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(
    "../data/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

def query_mistral(prompt: str) -> str:
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "mistral", "prompt": prompt, "stream": False}
    )
    return response.json()["response"]

def rag_pipeline(query: str, k: int = 3) -> dict:
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])
    sources = [f"{doc.metadata.get('source','?')} p.{doc.metadata.get('page','?')}" 
               for doc in docs]
    prompt = f"""You are a cybersecurity advisor helping practitioners.
Use ONLY the context below to answer the question.
If the context does not contain enough information, say so.

Context:
{context}

Question: {query}

Provide exactly 3 concise, actionable guidelines for cybersecurity practitioners.
Format as:
Guideline 1: ...
Guideline 2: ...
Guideline 3: ...
"""
    answer = query_mistral(prompt)
    return {
        "query": query,
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": [doc.page_content for doc in docs]
    }

# 跑所有queries
print("Running RAG pipeline for all queries...")
results = []
for i, query in enumerate(eval_queries):
    print(f"  [{i+1}/10] {query[:50]}...")
    result = rag_pipeline(query)
    results.append(result)

print("\nDone! All 10 queries completed.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Running RAG pipeline for all queries...
  [1/10] How should an organization respond to a phishing e...
  [2/10] What technical controls can prevent phishing attac...
  [3/10] How can employees identify spear-phishing attempts...
  [4/10] What is DMARC and how does it help prevent email s...
  [5/10] How should incident response teams handle a phishi...
  [6/10] What are best practices for email authentication i...
  [7/10] How can organizations train employees to recognize...
  [8/10] What role does SPF play in email security?...
  [9/10] How should a mid-sized enterprise handle a ransomw...
  [10/10] What are the indicators of compromise in a phishin...

Done! All 10 queries completed.


In [4]:
# 计算 RAGAS 评估指标
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from langchain_community.llms import Ollama
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import Dataset

# 增加超时时间
ollama_llm = LangchainLLMWrapper(Ollama(model="mistral", timeout=300))
hf_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)

# 减少并发，增加超时
run_config = RunConfig(
    timeout=300,
    max_retries=3,
    max_workers=1,
)

ragas_data = {
    "question": [r["query"] for r in results],
    "answer": [r["answer"] for r in results],
    "contexts": [r["retrieved_chunks"] for r in results],
}

dataset = Dataset.from_dict(ragas_data)

print("Running RAGAS evaluation with Ollama (this will take ~20 mins, please wait)...")
ragas_results = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=ollama_llm,
    embeddings=hf_embeddings,
    run_config=run_config,
)

print("\n=== RAGAS SCORES ===")
print(ragas_results)

/var/folders/tk/x43kyqbn33v_ttx5rckk29cm0000gn/T/ipykernel_3404/4180124688.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
/var/folders/tk/x43kyqbn33v_ttx5rckk29cm0000gn/T/ipykernel_3404/4180124688.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy
/var/folders/tk/x43kyqbn33v_ttx5rckk29cm0000gn/T/ipykernel_3404/4180124688.py:12: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/var/folders/tk/x43kyqbn33v_ttx5rckk29cm0000gn/T/ipykernel_3404/4180124688.py:13: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  hf_embeddings = LangchainEmbeddingsWrapper(


Running RAGAS evaluation with Ollama (this will take ~20 mins, please wait)...


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]


=== RAGAS SCORES ===
{'faithfulness': 0.8752, 'answer_relevancy': 0.6682}


In [6]:
import pandas as pd
import json

# 保存 RAGAS 分数表
scores_df = ragas_results.to_pandas()
scores_df.insert(0, "query", [r["query"] for r in results])
scores_df.to_csv("../outputs/ragas_scores.csv", index=False)
print("Scores saved!")
print(scores_df[["query", "faithfulness", "answer_relevancy"]].to_string())

# 保存完整结果
with open("../outputs/full_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nFull results saved!")

Scores saved!
                                                                                 query  faithfulness  answer_relevancy
0                              How should an organization respond to a phishing email?      0.428571          0.670904
1                                What technical controls can prevent phishing attacks?      0.666667          0.747638
2                                  How can employees identify spear-phishing attempts?      1.000000          0.366704
3                           What is DMARC and how does it help prevent email spoofing?      1.000000          0.563871
4            How should incident response teams handle a phishing-related data breach?      0.800000          0.786220
5                     What are best practices for email authentication in enterprises?      1.000000          0.717319
6                         How can organizations train employees to recognize phishing?      0.857143          0.627105
7                                 

In [ ]:
from bert_score import score as bert_score

# 用检索内容作为reference，答案作为candidate
candidates = [r["answer"] for r in results]
references = [" ".join(r["retrieved_chunks"]) for r in results]

P, R, F1 = bert_score(candidates, references, lang="en", verbose=False)

scores_df["bertscore_precision"] = P.numpy()
scores_df["bertscore_recall"] = R.numpy()
scores_df["bertscore_f1"] = F1.numpy()

scores_df.to_csv("../outputs/ragas_scores_with_bertscore.csv", index=False)

print("=== BERTScore Results ===")
print(scores_df[["query", "faithfulness", "answer_relevancy", "bertscore_f1"]].to_string())
print(f"\nMean BERTScore F1: {F1.mean():.4f}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]